# Tutorial: 3D Image Classification  

> SynapseMNIST3D dataset demo


In [1]:
#| default_exp demo

### Setup imports

In [1]:
from bioMONAI.data import *
from bioMONAI.transforms import *
from bioMONAI.core import *
from bioMONAI.core import Path
#from bioMONAI.data import get_image_files, get_target, RandomSplitter
from bioMONAI.losses import *
from bioMONAI.metrics import *
from bioMONAI.datasets import download_medmnist
from bioMONAI.visualize import show_images_grid, mosaic_image_3d


#from monai.utils import set_determinism
#from monai.transforms import ScaleIntensity

#set_determinism(0)

In [2]:
device = get_device()
print(device)

cuda


### Download and store the dataset

In [5]:
data_flag = 'synapsemnist3d'
data_path = Path('../_data/medmnist_data/')

info = download_medmnist(data_flag, data_path, download_only=True)

data_path = data_path/'synapsemnist3d'
train_path = data_path/'train'
val_path = data_path/'val'
test_path = data_path/'test'

Dataset 'synapsemnist3d' is already downloaded and available in '../_data/medmnist_data/synapsemnist3d'.


### Create Dataloader

In [6]:
from fastai.vision.all import CategoryBlock, get_image_files, GrandparentSplitter, parent_label
from monai.transforms import Resize

batch_size = 4

data_ops = {
    'blocks':       (BioImageBlock(cls=BioImageStack), CategoryBlock(info['label'])),
    'get_items':    get_image_files,
    'get_y':        parent_label,
    'splitter':     GrandparentSplitter(train_name='train', valid_name='val'), 
    'item_tfms':    [ScaleIntensity(), RandRot90(prob=0.5), Resize((32,32,32))],
    'bs':           batch_size,
}

data = BioDataLoaders.from_source(
    data_path, 
    show_summary=False,
    **data_ops,
    )

# print length of training and validation datasets
#print('train images:', len(data.train_ds.items), '\nvalidation images:', len(data.valid_ds.items))

In [ ]:
#im, label = data.train_ds[0]
#print(type(im), im.shape, label, label.shape)

<class 'bioMONAI.data.BioImageStack'> torch.Size([1, 28, 28, 28]) TensorCategory(0) torch.Size([])


In [ ]:
# visualization (ERROR PARA DATOS 3D)
#data.show_batch(max_n=6)

RuntimeError: a Tensor with 32768 elements cannot be converted to Scalar

### Load and train a 3D model

In [7]:
from monai.networks.nets import DenseNet121, HighResNet
from fastai.vision.all import BalancedAccuracy, CrossEntropyLossFlat, xresnet34
from torch.nn import CrossEntropyLoss

#model = HighResNet
model = DenseNet121(spatial_dims=3, in_channels=1, out_channels=1)
#model = xresnet34(n_out=2)

loss = CrossEntropyLossFlat()
metric = BalancedAccuracy

trainer = fastTrainer(data, model, loss_fn=loss, metrics=metric, show_summary=True)

DenseNet121 (Input shape: 4 x 1 x 32 x 32 x 32)
Layer (type)         Output Shape         Param #    Trainable 
                     4 x 64 x 16 x 16 x  
Conv3d                                    21952      True      
BatchNorm3d                               128        True      
ReLU                                                           
____________________________________________________________________________
                     4 x 64 x 8 x 8 x 8  
MaxPool3d                                                      
BatchNorm3d                               128        True      
ReLU                                                           
____________________________________________________________________________
                     4 x 128 x 8 x 8 x 8 
Conv3d                                    8192       True      
BatchNorm3d                               256        True      
ReLU                                                           
________________________________

In [ ]:
trainer.fit(20)

epoch,train_loss,valid_loss,BalancedAccuracy,time


ValueError: Expected input batch_size (4) to match target batch_size (131072).

In [ ]:
#trainer.show_results(cmap='gray')

In [ ]:
# trainer.save('tmp-model')

### Test data 
Evaluate the performance of the selected model on unseen data.
It’s important to not touch this data until you have fine tuned your model to get an unbiased evaluation!